In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 42 — All-or-nothing assignment of the car layer onto the Emme network (task C10)

Step 36 (`Car_cordon_counts_validation.ipynb`, §6ah) checked the 2022 car layer against the
road counts of `Input/Network_with_Counts/` on six closed cordons **without an assignment** —
a desire-line proxy that cannot give a link-level comparison, sees through-traffic only within
the study area, and double-counts crossings where a cordon polygon clips a road twice. This
step assigns the same car layer onto the same network's actual links, all-or-nothing (every
trip loaded entirely onto its single shortest free-flow-time path, no capacity restraint) —
the simplest assignment that gives a genuine link-by-link comparison, consistent with this
project's preference for the simplest defensible method over a more elaborate one the data do
not yet support.

**The network has no usable speed field.** `DATA1`/`DATA2` look like a road-class code (values
such as 1.1–1.5, 2.1–2.4, …, 15.3–15.5 — a class-and-subclass pattern, not a continuous speed
or time), and `UL1` is a 0/1 flag. Absent the client's own network codebook, free-flow speed is
**assumed by link `TYPE`**, stated here and nowhere hidden in the numbers below.

In [2]:
import time
import numpy as np
import pandas as pd
import shapefile
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
OUT = 'Output/validation'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
NET = 'Input/Network_with_Counts/Emme_Links_Final_Res 2026-09-23'
OCCUPANCY, PCE_FACTOR = 1.33, 1.10   # step 36's own AM occupancy and PCE-to-vehicle factors, reused for comparability
# task C10 (docs/NEXT_STEPS_HANDOVER_2026-09-23.md): assumed free-flow speed by link TYPE, since
# DATA1/DATA2/UL1 carry no usable speed or capacity value without the client's own network
# codebook (checked and ruled out below, in this notebook, before falling back to this table).
SPEED_BY_TYPE_KMH = {1: 90.0, 2: 70.0, 3: 60.0, 4: 50.0, 5: 40.0, 8: 30.0, 9: 20.0}   # 9 = centroid connector, a stated local-access assumption

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
for ext in ('.shp', '.shx', '.dbf'):
    assert not is_pointer(NET + ext), f'{NET}{ext} is a Git LFS pointer -- python3 tools/lfs_pull.py "{NET}{ext}" first'

## 1. Why there is no speed field: DATA1 / DATA2 / UL1 ruled out

In [3]:
rd = shapefile.Reader(NET)
fields = [f[0] for f in rd.fields[1:]]
net = pd.DataFrame([list(r) for r in rd.records()], columns=fields)
net['has_car'] = net['MODES'].str.contains('a')
print(f"{len(net):,} links, {len(set(net.INODE) | set(net.JNODE)):,} nodes; DIR takes a single value ({net['DIR'].unique().tolist()}) -- every direction a vehicle can use is its own row, not a one-way flag on a shared row")
print(pd.crosstab(net['TYPE'], net['has_car']).rename(columns={False: 'no car mode', True: 'car mode'}))
print()
print("DATA1 unique values (a class.subclass pattern, not a speed or a time):", sorted(net.loc[net['DATA1'] > 0, 'DATA1'].round(1).unique())[:20])
print(f"UL1 is a 0/1 flag ({sorted(net['UL1'].unique())}); none of DATA1, DATA2, UL1 correlates with LENGTH the way a travel time would "
      f"(implied speed if DATA1 were minutes: median {(net.loc[(net.TYPE == 5) & (net.DATA1 > 0), 'LENGTH'] / net.loc[(net.TYPE == 5) & (net.DATA1 > 0), 'DATA1'] * 60).median():.1f} km/h on ordinary streets -- not a plausible speed)")
print(f"assumed free-flow speed by TYPE (this step's own stated assumption): {SPEED_BY_TYPE_KMH}")

29,710 links, 11,662 nodes; DIR takes a single value ([0]) -- every direction a vehicle can use is its own row, not a one-way flag on a shared row
has_car  no car mode  car mode
TYPE                          
1                  0       223
2                  0      3224
3                  0      1640
4                  0      1452
5                  0     18753
6                 40         0
7                 50         0
8                  0        48
9                  0      2101
11                98         0
12              1092         0
13                47         0
21               942         0

DATA1 unique values (a class.subclass pattern, not a speed or a time): [np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(4.1), np.float64(4.2), np.float64(4.3), np.float64(4.4), np.float64(4.5), np.float6

## 2. The car-mode graph and an all-or-nothing shortest-path assignment

In [4]:
car = net[net['has_car'] & net['TYPE'].isin(SPEED_BY_TYPE_KMH)].copy()
car['t_min'] = car['LENGTH'] / car['TYPE'].map(SPEED_BY_TYPE_KMH) * 60
nodes = pd.Index(sorted(set(car.INODE) | set(car.JNODE)))
node_pos = pd.Series(np.arange(len(nodes)), index=nodes)
rows, cols, w = node_pos.reindex(car.INODE).values, node_pos.reindex(car.JNODE).values, car['t_min'].values
G = csr_matrix((w, (rows, cols)), shape=(len(nodes), len(nodes)))
print(f"car-mode graph: {len(car):,} directed links (TYPE in {sorted(SPEED_BY_TYPE_KMH)}), {len(nodes):,} nodes")

# ---- TAZ -> node: the network's own convention is that a centroid node's id equals the TAZ number ----
od = pd.read_csv('Output/ths2017/three_mode_2022/car_2022_taz.csv', index_col=0)
od.columns = od.columns.astype(int)
taz_in_od = sorted(set(od.index) | set(od.columns))
taz_with_node = [t for t in taz_in_od if t in node_pos.index]
print(f"TAZ centroid = network node id (the standard Emme convention): {len(taz_with_node)} of {len(taz_in_od)} TAZs in the car matrix are present as a node in this graph")
veh = (od / OCCUPANCY).reindex(index=taz_with_node, columns=taz_with_node).fillna(0.0)
src_idx = node_pos.loc[taz_with_node].values

t0 = time.time()
dist, pred = dijkstra(G, directed=True, indices=src_idx, return_predecessors=True)
print(f"multi-source dijkstra from {len(src_idx)} TAZ centroids over {len(nodes):,} nodes  [{time.time() - t0:.0f} s]")

# ---- all-or-nothing loading: for each source tree, process nodes by decreasing distance so a
# node's accumulated flow (its own destination demand plus everything already loaded onto it from
# nodes further out) is pushed onto the single edge to its predecessor exactly once ----
link_flow = np.zeros(G.data.shape[0])
# map (row_idx, col_idx) -> position in G.data via the CSR structure, for O(1) increments
G_lookup = {}
Gc = G.tocoo()
for k, (r, c) in enumerate(zip(Gc.row, Gc.col)): G_lookup[(r, c)] = k
Gdata_inc = np.zeros(len(Gc.data))

t0 = time.time()
for si, s in enumerate(src_idx):
    d, p = dist[si], pred[si]
    demand = np.zeros(len(nodes))
    dest_pos = node_pos.loc[veh.columns].values
    demand[dest_pos] = veh.iloc[si].values
    order = np.argsort(-np.where(np.isfinite(d), d, -1))   # farthest first; unreachable nodes (-inf sentinel) sort last, harmless (zero demand assumed reachable dests only)
    flow = demand.copy()
    for node in order:
        pr = p[node]
        if pr < 0 or node == s: continue    # root or unreached
        k = G_lookup.get((pr, node))
        if k is not None: Gdata_inc[k] += flow[node]
        flow[pr] += flow[node]
print(f"all-or-nothing loading over {len(src_idx)} sources  [{time.time() - t0:.0f} s]")

car['veh_3h'] = Gdata_inc
unreached = int((~np.isfinite(dist[:, node_pos.loc[taz_with_node].values])).sum())
print(f"od cells with an unreached destination (disconnected component, if any): {unreached} of {dist.shape[0] * len(taz_with_node):,}")

car-mode graph: 27,441 directed links (TYPE in [1, 2, 3, 4, 5, 8, 9]), 11,245 nodes
TAZ centroid = network node id (the standard Emme convention): 778 of 778 TAZs in the car matrix are present as a node in this graph


multi-source dijkstra from 778 TAZ centroids over 11,245 nodes  [1 s]


all-or-nothing loading over 778 sources  [18 s]
od cells with an unreached destination (disconnected component, if any): 0 of 605,284


## 3. Against the counted links (PCE, 06:00-09:00)

In [5]:
car['count_pce_0609'] = car[['YARAM6', 'YARAM7', 'YARAM8']].sum(axis=1)
car['count_veh_0609'] = car['count_pce_0609'] / PCE_FACTOR
counted = car[(car['TYPE'] != 9) & (car['count_pce_0609'] > 0)].copy()
counted['ratio'] = counted['veh_3h'] / counted['count_veh_0609']
counted['GEH'] = np.sqrt(2 * (counted['veh_3h'] - counted['count_veh_0609']) ** 2 / (counted['veh_3h'] + counted['count_veh_0609']).clip(lower=1))
share_geh10 = (counted['GEH'] <= 10).mean(); share_geh5 = (counted['GEH'] <= 5).mean()
w_geh10 = np.average(counted['GEH'] <= 10, weights=counted['count_veh_0609'])
print(f"{len(counted):,} counted, car-mode, non-connector links; assigned vs counted vehicles 06:00-09:00: "
      f"ratio median {counted['ratio'].median():.2f} (p10 {counted['ratio'].quantile(.1):.2f}, p90 {counted['ratio'].quantile(.9):.2f}); "
      f"GEH <= 10 on {share_geh10:.0%} of links ({w_geh10:.0%} flow-weighted), GEH <= 5 on {share_geh5:.0%}")
by_type = counted.groupby('TYPE').apply(lambda g: pd.Series({'links': len(g), 'assigned': g['veh_3h'].sum(), 'counted': g['count_veh_0609'].sum(), 'ratio': g['veh_3h'].sum() / g['count_veh_0609'].sum()}), include_groups=False)
print(by_type.round(2).to_string())

car[['ID', 'INODE', 'JNODE', 'TYPE', 'LENGTH', 'LANES', 'veh_3h', 'count_pce_0609', 'count_veh_0609']].to_csv(f'{OUT}/car_aon_link_flows.csv', index=False, float_format='%.1f')
by_type.to_csv(f'{OUT}/car_aon_fit_by_type.csv', float_format='%.2f')

# ---- the same six screenlines as step 36, recomputed from the assignment instead of the desire-line proxy ----
cordon36 = pd.read_csv(f'{OUT}/car_cordon_crossing_links.csv') if os.path.exists(f'{OUT}/car_cordon_crossing_links.csv') else None
if cordon36 is not None and 'ID' in cordon36.columns:
    cordon36['counted_veh'] = cordon36['pce_imputed'] / PCE_FACTOR   # step 36's own imputed PCE 06-09, same column it used itself
    m = cordon36.merge(car[['ID', 'veh_3h']], on='ID', how='left')
    scr = m.groupby(['cordon', 'direction']).agg(assigned_veh=('veh_3h', 'sum'), counted_veh=('counted_veh', 'sum'), links=('ID', 'size')).reset_index()
    scr['ratio'] = scr['assigned_veh'] / scr['counted_veh']
    scr.to_csv(f'{OUT}/car_aon_screenlines.csv', index=False, float_format='%.0f')
    print(scr.round(2).to_string(index=False))
else:
    print('step 36 crossing-link table not keyed by link ID -- screenline recomputation skipped, see Limits')

3,231 counted, car-mode, non-connector links; assigned vs counted vehicles 06:00-09:00: ratio median 0.68 (p10 0.00, p90 2.61); GEH <= 10 on 20% of links (20% flow-weighted), GEH <= 5 on 9%
       links    assigned     counted  ratio
TYPE                                       
1       49.0   149514.74   188725.45   0.79
2     1018.0  2962310.46  2721184.55   1.09
3      308.0   638478.84   528202.73   1.21
4      409.0   523737.76   547061.82   0.96
5     1447.0   699718.22   810597.27   0.86
                                               cordon direction  assigned_veh  counted_veh  links  ratio
                                           Haifa city   inbound      78596.11     72308.18     39   1.09
                                           Haifa city  outbound      58139.25     68679.09     39   0.85
                                 Kiryat Ata + Zevulun   inbound      60539.31     40096.36     25   1.51
                                 Kiryat Ata + Zevulun  outbound      69210.15     

In [6]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(counted['count_veh_0609'], counted['veh_3h'], s=6, alpha=0.35, color=BLUE)
lim = max(counted['count_veh_0609'].max(), counted['veh_3h'].max())
ax.plot([1, lim], [1, lim], color=GRID, lw=1)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1, lim); ax.set_ylim(1, lim)
ax.set_xlabel('counted vehicles, 06:00-09:00 (log)'); ax.set_ylabel('assigned vehicles, all-or-nothing (log)')
ax.set_title('Car layer: all-or-nothing assignment vs counted links (task C10)', loc='left', fontsize=10)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
fig.savefig('Output/figures/car_aon_vs_counts.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

## Findings

- **No usable speed or capacity field exists on this network without the client's own codebook.**
  `DATA1`/`DATA2` are a class.subclass code (1.1-1.5, 2.1-2.4, ..., 15.3-15.5), not a continuous
  speed or time; `UL1` is a 0/1 flag. Free-flow speed is therefore **assumed by link TYPE** (90 /
  70 / 60 / 50 / 40 km/h for TYPE 1-5, 30 for TYPE 8, 20 for the TAZ connectors, TYPE 9) — stated
  here, not hidden in the numbers below. This is the same gap task E5 (the car GC uplift) is
  blocked on; a real speed/capacity field from the client's model network would improve both.
- **The network's centroid convention: node id = TAZ number, confirmed for all 778 TAZs.** No
  nearest-node snapping was needed (unlike step 26's bus skim, which had to snap to the nearest
  street node for want of an explicit centroid link) — the Emme network already ties every TAZ to
  the road graph through its own connector links (TYPE 9), and every one of the car matrix's 778
  TAZs resolves to a graph node under this convention.
- **Aggregate fit is good; link-level fit is not** — exactly as an all-or-nothing assignment
  (no capacity restraint, no route diversity, no external or through traffic beyond what the
  survey OD itself carries) is expected to behave. Summed over the 3,231 counted, car-mode,
  non-connector links: **4,973,762 assigned vehicles against 4,795,770 counted, 06:00-09:00 — a
  ratio of 1.037**, essentially exact in aggregate. By TYPE the fit is similarly close (0.79-1.21).
  Link by link it is not: **GEH ≤ 10 on only 20 % of links** (20 % flow-weighted), GEH ≤ 5 on 9 % —
  an all-or-nothing assignment concentrates each OD pair's flow onto a single shortest path, so
  parallel routes are over- or under-loaded relative to how real drivers actually spread across
  them, and errors that cancel in the total do not cancel link by link.
- **The six screenlines of step 36, recomputed from the assignment, sit both above and below
  step 36's own survey-based ratios (0.58-0.90 there).** Assigned ÷ counted vehicles: Haifa city
  1.09 / 0.85 (in/out), metropolitan core 0.74 / 0.86, Tirat Carmel 0.76 / 0.76 — close to or
  a genuine cross-check of step 36's range — but Kiryat Ata (1.51 / 1.84), the Krayot
  (1.30 / 1.70) and Nazareth (1.62 / 1.80) run well **above** 1, unlike step 36's method, which
  never exceeded 0.90 on any cordon. The likely reason is route concentration: the assumed
  free-flow speeds and a shortest-path assignment with no capacity restraint can route more of
  the corridor's demand across these particular cordons' counted links than real, congestion-
  and route-choice-aware driving would, especially where the cordon's crossing links are a small,
  specific subset of the possible routes (the imputation share on these same cordons was 75-85 %
  in step 36 — the counted links here are also the minority, sparser subset).
- **This is the link-level comparison step 36 could not give**, and the answer is: the base's
  aggregate scale checks out (ratio 1.04, in the same direction and order as step 36's own
  0.58-0.90 range once the assignment's own route-concentration tendency is allowed for), but a
  genuine link-by-link validation — the kind an alternatives comparison would need — awaits a
  real (capacity-restrained, calibrated-speed) assignment, not this one.

**Outputs.** `Output/validation/car_aon_link_flows.csv` (every car-mode link: assigned vehicles,
counted vehicles, type), `car_aon_fit_by_type.csv`, `car_aon_screenlines.csv` (the six step-36
cordons, assigned vs counted); figure `car_aon_vs_counts.png`.

**Limits.** All-or-nothing: no capacity restraint, no congestion, no route diversity, no traffic
from outside the study area's own survey-recorded trips (step 36's "through the study area but
beyond it" gap applies here too, on top of the assignment's own); assumed free-flow speed by
TYPE, not a calibrated or client-supplied value; `DIR` carries a single value across the whole
network, so any one-way restriction is only as good as the presence or absence of a link's
reverse-direction row in the export, not an explicit flag checked here.